In [37]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict

In [ ]:
class BatsmanState(TypedDict):
    runs:int
    balls:int
    fours:int
    sixes:int

    sr:float
    bpb:float
    boundary_percent:float

    summary:str

In [39]:
def strike_rate(state: BatsmanState) -> BatsmanState:
    runs = state["runs"]
    balls = state["balls"]
    sr = (runs / balls) * 100

    return {"sr": sr}

In [40]:
def bpb(state: BatsmanState) -> BatsmanState:
    bpb = state["balls"] / (state["fours"] + state["sixes"])
    return {"bpb": bpb}

In [41]:
def boundary_percent(state: BatsmanState) -> BatsmanState:
    boundary_percent = (
        ((state["fours"] * 4) + (state["sixes"] * 6)) / state["runs"]
    ) * 100

    return {"boundary_percent": boundary_percent}

In [42]:
def summary(state: BatsmanState) -> BatsmanState:
    summary = f"""
    Strike Rate - {state['sr']} \n
    Balls per boundary - {state['bpb']} \n
    Boundary Percent - {state['boundary_percent']}
"""
    state['summary'] = summary
    return state

In [43]:
graph = StateGraph(BatsmanState)

graph.add_node('strike_rate',strike_rate)
graph.add_node('boundary_percent',boundary_percent)
graph.add_node('bpb',bpb)
graph.add_node('summary',summary)

graph.add_edge(START,'strike_rate')
graph.add_edge(START,'bpb')
graph.add_edge(START,'boundary_percent')

graph.add_edge('strike_rate','summary')
graph.add_edge('bpb','summary')
graph.add_edge('boundary_percent','summary')
graph.add_edge('summary',END)

workflow = graph.compile()



In [44]:
initial_state = {'runs':51,'balls':26,'fours':6,'sixes':1}
final_state = workflow.invoke(initial_state)
print(final_state)



{'runs': 51, 'balls': 26, 'fours': 6, 'sixes': 1, 'sr': 196.15384615384613, 'bpb': 3.7142857142857144, 'boundary_percent': 58.82352941176471, 'summary': '\n    Strike Rate - 196.15384615384613 \n\n    Balls per boundary - 3.7142857142857144 \n\n    Boundary Percent - 58.82352941176471\n'}
